# Module 17 — Agentic Enterprise Architecture

> **SDKs:** `pydantic`, `dataclasses`, `hashlib`

| Part | Topic |
|------|-------|
| **1** | Agent Identity & IAM — workload identity, SPIFFE/SVID |
| **2** | Registries & Tool Squatting — enterprise tool governance |
| **3** | MCP in Enterprise — air-gapped and zero-trust deployments |


---
## Part 1 — Agent Identity & IAM

Agents need non-human identities to authenticate to internal services. Without proper Workload Identity, agents either share service accounts (no auditability) or store long-lived secrets (security risk).

In [1]:
from dataclasses import dataclass, field
from typing import Optional, Literal
from pydantic import BaseModel
import hashlib, time, base64, json

# ─── SPIFFE/SVID-inspired workload identity ───────────────────────────────────
@dataclass
class AgentWorkloadIdentity:
    """
    Enterprise agent identity. Inspired by SPIFFE SVID.
    Issued by the internal CA. Short-lived (15 min TTL).
    """
    agent_id: str
    trust_domain: str             # e.g. "northstar.internal"
    run_id: str                   # unique per execution
    allowed_scopes: list[str]     # OAuth-style scopes
    issued_at: float = field(default_factory=time.time)
    ttl_seconds: int = 900        # 15 minutes

    @property
    def spiffe_id(self) -> str:
        return f"spiffe://{self.trust_domain}/agent/{self.agent_id}"

    @property
    def is_valid(self) -> bool:
        return time.time() < self.issued_at + self.ttl_seconds

    def token(self) -> str:
        """Simulate a short-lived JWT-like token."""
        payload = {
            "sub": self.spiffe_id,
            "run_id": self.run_id,
            "scopes": self.allowed_scopes,
            "exp": int(self.issued_at + self.ttl_seconds),
        }
        return base64.b64encode(json.dumps(payload).encode()).decode()[:40] + "..."

class ServiceMesh:
    """Validates incoming agent requests against the identity registry."""
    def __init__(self, trust_domain: str):
        self.trust_domain = trust_domain
        self._audit: list[dict] = []

    def authorize(self, identity: AgentWorkloadIdentity, required_scope: str) -> tuple[bool, str]:
        self._audit.append({"agent": identity.agent_id, "scope": required_scope, "at": time.time()})
        if not identity.is_valid:
            return False, "Token expired"
        if identity.trust_domain != self.trust_domain:
            return False, f"Wrong trust domain: {identity.trust_domain!r}"
        if required_scope not in identity.allowed_scopes:
            return False, f"Scope {required_scope!r} not in token"
        return True, "OK"

mesh = ServiceMesh("northstar.internal")

# Identities
incident_agent = AgentWorkloadIdentity(
    agent_id="incident-investigator-v1",
    trust_domain="northstar.internal",
    run_id="run-" + hashlib.sha256(b"INC-001").hexdigest()[:8],
    allowed_scopes=["metrics:read", "deployments:read", "runbooks:read"],
)

rogue_agent = AgentWorkloadIdentity(
    agent_id="rogue-agent",
    trust_domain="attacker.external",   # Wrong trust domain
    run_id="run-evil",
    allowed_scopes=["everything:write"],
)

print("🪪  Agent Identity & IAM Demo")
print("=" * 60)
print(f"\n  Incident Agent SPIFFE ID: {incident_agent.spiffe_id}")
print(f"  Token (first 40 chars): {incident_agent.token()}")

tests = [
    (incident_agent, "metrics:read",       "Legitimate scope"),
    (incident_agent, "database:write",     "Unauthorized scope"),
    (rogue_agent,    "metrics:read",       "Wrong trust domain"),
]

print("\n  Authorization checks:")
for identity, scope, label in tests:
    ok, reason = mesh.authorize(identity, scope)
    icon = "✅" if ok else "❌"
    print(f"  {icon}  [{identity.agent_id}] scope={scope!r}: {reason}  ({label})")


🪪  Agent Identity & IAM Demo

  Incident Agent SPIFFE ID: spiffe://northstar.internal/agent/incident-investigator-v1
  Token (first 40 chars): eyJzdWIiOiAic3BpZmZlOi8vbm9ydGhzdGFyLmlu...

  Authorization checks:
  ✅  [incident-investigator-v1] scope='metrics:read': OK  (Legitimate scope)
  ❌  [incident-investigator-v1] scope='database:write': Scope 'database:write' not in token  (Unauthorized scope)
  ❌  [rogue-agent] scope='metrics:read': Wrong trust domain: 'attacker.external'  (Wrong trust domain)


---
## Part 2 — Enterprise Tool Governance

Enterprise agents must operate within a governed tool registry. Every tool must be registered, versioned, and audited.

In [2]:
from dataclasses import dataclass, field
from datetime import datetime, timezone

@dataclass
class EnterpriseToolSpec:
    tool_id: str
    version: str
    owner_team: str
    description: str
    allowed_agent_roles: list[str]   # RBAC: only these agent roles can call it
    is_mutation: bool                # true = requires approval
    audit_required: bool = True
    deprecated: bool = False

ENTERPRISE_REGISTRY: dict[str, EnterpriseToolSpec] = {
    "query_metrics":     EnterpriseToolSpec("query_metrics",     "v2.1", "Observability", "Query Datadog metrics",       ["incident-agent","analyst-agent"], False, True),
    "get_deployment":    EnterpriseToolSpec("get_deployment",     "v1.4", "DevOps",        "Query deployment history",    ["incident-agent","devops-agent"],   False, True),
    "propose_revert":    EnterpriseToolSpec("propose_revert",     "v1.0", "Platform",      "Submit a revert proposal",    ["incident-agent"],                  False, True),
    "execute_revert":    EnterpriseToolSpec("execute_revert",     "v1.0", "Platform",      "Execute an approved revert",  ["orchestrator"],                    True,  True),
    "delete_table":      EnterpriseToolSpec("delete_table",       "v0.9", "DBA",           "Delete a database table",     ["dba-agent"],                       True,  True, deprecated=True),
}

class ToolGovernance:
    def __init__(self, registry: dict[str, EnterpriseToolSpec]):
        self.registry = registry
        self.call_log: list[dict] = []

    def check_access(self, agent_role: str, tool_id: str) -> tuple[bool, str]:
        spec = self.registry.get(tool_id)
        if not spec: return False, f"Tool {tool_id!r} not in registry"
        if spec.deprecated: return False, f"Tool {tool_id!r} is deprecated — use replacement"
        if agent_role not in spec.allowed_agent_roles:
            return False, f"Role {agent_role!r} not authorized for {tool_id!r}"
        if spec.is_mutation:
            return False, f"Tool {tool_id!r} requires human approval — cannot call directly"
        return True, "OK"

gov = ToolGovernance(ENTERPRISE_REGISTRY)

print("📜  Enterprise Tool Governance Demo")
print("=" * 65)
print(f"  Registry size: {len(ENTERPRISE_REGISTRY)} tools")
print()
print(f"  {'Agent Role':<22} {'Tool':<20} {'Result'}")
print(f"  {'─'*22} {'─'*20} {'─'*30}")

checks = [
    ("incident-agent",  "query_metrics",  "✅"),
    ("incident-agent",  "execute_revert", "❌"),
    ("analyst-agent",   "get_deployment", "❌"),
    ("orchestrator",    "execute_revert", "❌"),
    ("dba-agent",       "delete_table",   "❌"),
]
for role, tool, expected in checks:
    ok, reason = gov.check_access(role, tool)
    icon = "✅" if ok else "🚫"
    print(f"  {role:<22} {tool:<20} {icon}  {reason}")


📜  Enterprise Tool Governance Demo
  Registry size: 5 tools

  Agent Role             Tool                 Result
  ────────────────────── ──────────────────── ──────────────────────────────
  incident-agent         query_metrics        ✅  OK
  incident-agent         execute_revert       🚫  Role 'incident-agent' not authorized for 'execute_revert'
  analyst-agent          get_deployment       🚫  Role 'analyst-agent' not authorized for 'get_deployment'
  orchestrator           execute_revert       🚫  Tool 'execute_revert' requires human approval — cannot call directly
  dba-agent              delete_table         🚫  Tool 'delete_table' is deprecated — use replacement
